In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import plotly.express as px
import plotly.io as pio
import pandas as pd

from turbofan.config.schema import load_config
from turbofan.data.loader import load_raw_train, load_raw_test, load_rul_labels
from turbofan.eda import quality, sensors, degradation

pio.templates.default = "plotly_white"
plt.rcParams["figure.figsize"] = (12, 6)

if Path.cwd().name == "notebooks":
    %cd ..

cfg = load_config(Path("configs/default.yaml"))
cfg = cfg.model_copy(
    update={"data": cfg.data.model_copy(update={"fd_subset": "FD004"})}
)
print(cfg.data.fd_subset)

In [ ]:
train_df = load_raw_train(cfg.data)
test_df  = load_raw_test(cfg.data)
test_rul = load_rul_labels(cfg.data)

print(f"Train: {train_df.shape[0]:,} rows, {train_df['engine_id'].nunique()} engines")
print(f"Test:  {test_df.shape[0]:,} rows,  {test_df['engine_id'].nunique()} engines")
print(f"RUL labels: {len(test_rul)} engines")
train_df.head()

## 1. Data Quality

In [ ]:
missing = quality.find_missing_values(train_df)
print("Missing values per column:")
print(missing[missing > 0] if missing.any() else "None — dataset is complete.")

In [ ]:
constant = quality.find_constant_sensors(train_df)
sensor_cols = [c for c in train_df.columns if c.startswith("s_")]
non_constant = [c for c in sensor_cols if c not in constant]

print(f"Constant sensors ({len(constant)}): {constant}")
print(f"Non-constant sensors ({len(non_constant)}): {non_constant}")

## 2. Operational Settings

In [ ]:
op_cols = ["op_1", "op_2", "op_3"]

op_combos = (
    train_df[op_cols]
    .round(0)
    .groupby(op_cols)
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)
print(f"Operating condition combinations: {len(op_combos)}")
op_combos

## 3. RUL Labels

In [ ]:
train_with_rul = degradation.compute_rul_curves(train_df, max_rul=cfg.data.max_rul)
rul = train_with_rul["rul"]

lifetimes = train_df.groupby("engine_id")["cycle"].max()
print(f"Engine lifetime — min: {lifetimes.min()}, median: {lifetimes.median():.0f}, max: {lifetimes.max()}")

fig = px.histogram(
    lifetimes,
    nbins=30,
    labels={"value": "Max cycle", "count": "Engines"},
    title="Engine Lifetime Distribution",
)
fig.show()

## 4. Sensor Stats

In [ ]:
stats = sensors.compute_sensor_stats(train_df)
stats.loc[non_constant].round(3)

## 5. Operating-Mode Normalization

Required before correlation filtering: 6 operating conditions dominate raw sensor variance and mask the degradation signal.

In [ ]:
from turbofan.preprocessing.normalization import OperatingModeNormalizer

normalizer = OperatingModeNormalizer(
    feature_cols=non_constant,
    op_cols=op_cols,
    n_modes=6,
    random_state=cfg.data.random_seed,
)
normalizer.fit(train_df)
train_norm = normalizer.transform(train_df)

centers_df = pd.DataFrame(normalizer.mode_centers_, columns=op_cols).round(2)
centers_df.index.name = "mode"
print("Discovered mode centres:")
print(centers_df)
print()
print("Known operating combos:")
print(op_combos.drop(columns="count").to_string(index=False))

In [ ]:
post_std = train_norm[non_constant].std()
new_zero = sorted(post_std[post_std < 1e-6].index.tolist())
print(f"Zero-variance after normalization (constant within every mode): {new_zero}")

## 6. Correlation Filter

In [ ]:
CORR_THRESHOLD = 0.1

rul_corr_raw  = train_df[non_constant].corrwith(rul).sort_values()
rul_corr      = train_norm[non_constant].corrwith(rul).sort_values()

comparison = pd.DataFrame({"raw": rul_corr_raw, "normalized": rul_corr}).sort_values("raw")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, col, title in zip(axes, ["raw", "normalized"], ["Raw", "Per-mode normalized"]):
    ax.barh(comparison.index, comparison[col])
    ax.axvline(0, color="black", linewidth=0.8)
    ax.axvline(CORR_THRESHOLD,  color="red", linestyle="--", linewidth=0.8)
    ax.axvline(-CORR_THRESHOLD, color="red", linestyle="--", linewidth=0.8)
    ax.set_xlabel("Pearson r with RUL")
    ax.set_title(f"Sensor-RUL Correlation — {title}")
plt.tight_layout()
plt.show()

informative = rul_corr[rul_corr.abs() >= CORR_THRESHOLD].index.tolist()
dropped     = rul_corr[rul_corr.abs() <  CORR_THRESHOLD].index.tolist()

print(f"Informative ({len(informative)}): {informative}")
print(f"Dropped     ({len(dropped)}):     {dropped}")

## 6. Low-Variance Check

Among informative sensors, flag any with low absolute std — high correlation despite low spread warrants inspection.

In [ ]:
low_var = quality.find_low_variance_sensors(train_df[informative], tol=1e-2)

if low_var:
    print(f"Low-variance but informative sensors: {low_var}")
    print(train_df[low_var].describe().round(6))
else:
    print("No low-variance sensors among informative set.")

## 8. Sensor Distributions (before/after normalization)

Top 4 sensors by post-normalization |corr with RUL|.

In [ ]:
top_sensors = rul_corr.abs().sort_values(ascending=False).index.tolist()

fig, axes = plt.subplots(len(top_sensors), 2, figsize=(14, 4 * len(top_sensors)))
for row, sensor in enumerate(top_sensors):
    train_df[sensor].hist(bins=40, ax=axes[row, 0], edgecolor="black", alpha=0.7)
    train_norm[sensor].hist(bins=40, ax=axes[row, 1], edgecolor="black", alpha=0.7)
    axes[row, 0].set_title(f"{sensor} — raw")
    axes[row, 1].set_title(f"{sensor} — normalized")
plt.suptitle("Sensor distributions before and after normalization", y=1.02)
plt.tight_layout()
plt.show()

## 9. Degradation Trajectories (normalized)

In [ ]:
sample_engines = sorted(train_df["engine_id"].unique())[:5]
sample_norm = train_norm[train_df["engine_id"].isin(sample_engines)].copy()

fig, axes = plt.subplots(len(top_sensors), 1, figsize=(14, 4 * len(top_sensors)))
if len(top_sensors) == 1:
    axes = [axes]
for ax, sensor in zip(axes, top_sensors):
    for eid in sample_engines:
        d = sample_norm[sample_norm["engine_id"] == eid]
        ax.plot(d["cycle"], d[sensor], alpha=0.7, label=f"Engine {eid}")
    ax.set_ylabel(f"{sensor} (normalized)")
    ax.legend(loc="upper left", fontsize=8)
axes[-1].set_xlabel("Cycle")
plt.suptitle("Normalized Sensor Degradation Trajectories (sample engines)", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
smoothed = degradation.compute_sensor_trends(sample_norm, top_sensors, window=10)

fig, axes = plt.subplots(len(top_sensors), 1, figsize=(14, 4 * len(top_sensors)))
if len(top_sensors) == 1:
    axes = [axes]
for ax, sensor in zip(axes, top_sensors):
    for eid in sample_engines:
        d = smoothed[smoothed["engine_id"] == eid]
        ax.plot(d["cycle"], d[sensor], alpha=0.7, label=f"Engine {eid}")
    ax.set_ylabel(f"{sensor} (smoothed)")
    ax.legend(loc="upper left", fontsize=8)
axes[-1].set_xlabel("Cycle")
plt.suptitle("Smoothed Normalized Sensor Trends (window=10)", y=1.01)
plt.tight_layout()
plt.show()

## 10. Summary

**Dataset:** 61,249 rows, 249 engines, 6 operating conditions

**Data quality:** No missing values. No constant sensors.

**Engine lifetimes:** min 128, median 234, max 543 cycles — longest across all subsets.

**Normalization:** `OperatingModeNormalizer` (n_modes=6) recovered all 6 known operating points. Zero-variance after normalization: `s_1`, `s_5`, `s_18`, `s_19`.

**Raw correlation:** Nearly all sensors below threshold. Only `s_14` passes on raw data.

**Sensor filter post-normalization (|corr| ≥ 0.1):**
- Informative (14): `s_2`, `s_3`, `s_4`, `s_6`, `s_7`, `s_8`, `s_9`, `s_10`, `s_11`, `s_12`, `s_13`, `s_14`, `s_16`, `s_17`
- Dropped: `s_1`, `s_5`, `s_15`, `s_18`, `s_19`, `s_20`, `s_21`

**Low-variance check:** `s_16` is binary (0.02 / 0.03) — correlation may reflect an operating state switch rather than degradation; treat with caution.
